# Credit Card Fraud Detection

End-to-end walkthrough: loading the data, handling class imbalance, training models, and evaluating with the right metrics.

The data is from the [Kaggle ULB dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). Place `creditcard.csv` in `/data` before running — or let the loader generate synthetic data automatically.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.features import engineer_features, get_feature_columns
from src.balancer import resample
from src.models import get_models, cross_validate_model, train_final
from src.evaluate import evaluate, tune_threshold, plot_roc_pr_curves
from sklearn.model_selection import train_test_split

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Load the data

In [ ]:
df = load_data()
df.head()

## 2. Class imbalance — the core challenge

This is why accuracy is a useless metric here. A model that predicts 'legit' for everything scores ~99.8% accuracy while catching zero fraud.

In [ ]:
class_counts = df['Class'].value_counts()
print(f"Legit transactions : {class_counts[0]:,}")
print(f"Fraud transactions : {class_counts[1]:,}")
print(f"Fraud rate         : {class_counts[1]/len(df)*100:.3f}%")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# raw counts
axes[0].bar(['Legit', 'Fraud'], class_counts.values, color=['#2196F3', '#FF5722'])
axes[0].set_title('Transaction counts (log scale)')
axes[0].set_yscale('log')
axes[0].set_ylabel('Count (log)')

# pie — shows just how tiny the fraud slice is
axes[1].pie(class_counts.values, labels=['Legit', 'Fraud'],
            colors=['#2196F3', '#FF5722'], autopct='%1.3f%%',
            startangle=90, pctdistance=0.7)
axes[1].set_title('Class proportion')
plt.tight_layout()

## 3. Feature distributions

V1–V28 are PCA-compressed and confidential. `Amount` and `Time` are interpretable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Amount distribution by class
for cls, color, label in [(0, '#2196F3', 'Legit'), (1, '#FF5722', 'Fraud')]:
    subset = df[df['Class'] == cls]['Amount']
    axes[0].hist(subset, bins=60, alpha=0.6, color=color, label=label, log=True)
axes[0].set_title('Transaction Amount by Class')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count (log)')
axes[0].legend()

# Hour of day (derived from Time)
df_plot = df.copy()
df_plot['hour'] = (df_plot['Time'] % 86400) // 3600
fraud_by_hour  = df_plot[df_plot['Class']==1].groupby('hour').size()
legit_by_hour  = df_plot[df_plot['Class']==0].groupby('hour').size() / 500  # scaled

axes[1].bar(fraud_by_hour.index, fraud_by_hour.values, color='#FF5722', label='Fraud (actual count)')
axes[1].plot(legit_by_hour.index, legit_by_hour.values, 'b--', label='Legit (÷500, scaled)')
axes[1].set_title('Activity by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Fraud count')
axes[1].legend()
plt.tight_layout()

## 4. Feature engineering

In [ ]:
df = engineer_features(df)
feature_cols = get_feature_columns(df)
print(f"Features ({len(feature_cols)}): {feature_cols[:8]} ... {feature_cols[-3:]}")

## 5. Train/test split

In [ ]:
X = df[feature_cols].values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")
print(f"Fraud in test set: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

## 6. SMOTE + undersampling — balancing the training set

Important: we only resample the *training* set. The test set stays as-is to reflect real-world conditions.

In [ ]:
from collections import Counter

X_res, y_res = resample(X_train, y_train, strategy='combined')

print("Before resampling:", Counter(y_train))
print("After  resampling:", Counter(y_res))

## 7. Cross-validation (resampling inside each fold)

In [ ]:
models = get_models()
rf = models['random_forest']

print("Random Forest — 5-fold stratified CV")
cv_scores = cross_validate_model(rf, X_train, y_train, resample_strategy='combined')
print(f"\nAUC-ROC: {cv_scores['auc_roc'].mean():.4f} ± {cv_scores['auc_roc'].std():.4f}")
print(f"AUC-PR : {cv_scores['auc_pr'].mean():.4f} ± {cv_scores['auc_pr'].std():.4f}")

## 8. Train all models and evaluate

In [ ]:
models = get_models()
results = {}

for name, model in models.items():
    model = train_final(model, X_train, y_train)
    res   = evaluate(model, X_test, y_test, model_name=name)
    results[name] = res

## 9. ROC vs PR curves — why PR tells the real story

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

palette = ['#2196F3', '#4CAF50', '#FF5722']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ROC vs Precision-Recall — notice how PR is more discriminating', fontsize=12)

for i, (name, res) in enumerate(results.items()):
    probs = res['probs']
    fpr, tpr, _ = roc_curve(y_test, probs)
    prec, rec, _ = precision_recall_curve(y_test, probs)

    axes[0].plot(fpr, tpr, color=palette[i], lw=1.8,
                 label=f"{name} (AUC={res['auc_roc']:.3f})")
    axes[1].plot(rec, prec, color=palette[i], lw=1.8,
                 label=f"{name} (AP={res['auc_pr']:.3f})")

axes[0].plot([0,1],[0,1],'k--',lw=0.8); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].axhline(y_test.mean(), color='k', linestyle='--', lw=0.8, label='Random')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()

## 10. Threshold tuning — optimising for business cost

Default 0.5 threshold is rarely optimal. Missed fraud costs ~$120; a false alarm costs ~$2.

In [ ]:
# use the best model from results
best_name = max(results, key=lambda k: results[k]['auc_pr'])
print(f"Best model by AUC-PR: {best_name}")

best_thresh = tune_threshold(results[best_name]['probs'], y_test, model_name=best_name)

## Summary

For production, prefer the saved `train.py` CLI — it handles all of the above automatically and saves models to `outputs/models/` for later use with `predict.py`.